# Misinformation Models—
Driver notebook: trains all models to date and saves them.

### Import configuration and packages ###

In [ ]:
##TO RUN ON COLAB ONLY


#installs
!pip install tweet-preprocessor==0.5.0 feedparser whoosh iterative-stratification fastapi uvicorn
!pip install nltk spacy

#imports
import sys
from pathlib import Path
from google.colab import drive
import pandas as pd
import numpy as np
import torch

#mount google drive and set path-related variables.
drive.mount('/content/drive')
BASE_DIR=Path("/content/drive/MyDrive/linguistic_markers")
SPRINT_DIR = BASE_DIR / "581_Sprint_4"
SRC_DIR = SPRINT_DIR / "src"
DATA_DIR = BASE_DIR / "data" / "final_splits"

# Add the SPRINT_DIR to the system path so Python can find modules like 'cnn_baseline'
sys.path.insert(0, str(Path.cwd()))
sys.path.insert(1, str(SPRINT_DIR))
sys.path.insert(2, str(SRC_DIR))
sys.path.insert(3, str(DATA_DIR))
sys.path.insert(4, str(BASE_DIR))

# Define the device for training
DEVICE = torch.device("cuda" + ":0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


import cnn_baseline as cnn
#Running this will import FastText vector file, stored on HuggingFace and is >4gb.
from config import FASTTEXT_PATH, TARGETS, SEED
from preprocess import preprocess
from metrics import compute_metrics, print_confusion_matrix, print_sklearn_report, error_analysis, print_report
import random


random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)


  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 468.8/468.8 kB 19.0 MB/s eta 0:00:00
  Created wheel for tweet-preprocessor: filename=tweet_preprocessor-0.5.0-py3-none-any.whl size=7928 sha256=800f7bb8ddb606f2f8ce1fc541d201db5a864fd13d237593744ecce94f59d6c3
  Stored in directory: /root/.cache/pip/wheels/4b/6e/04/d26d41ed041dd0318b112367564452d05a4835d1a3f8e37518
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6046 sha256=aac90ddaa985aaa62e0171f533bf57074aaaa4d9291930d0ef75e071c66c8432
  Stored in directory: /root/.cache/pip/wheels/03/f5/1a/23761066dac1d0e8e683e5fdb27e12de53209d05a4a37e6246
Successfully built tweet-preprocessor sgmllib3k
Mounted at /content/drive
Using device: cuda:0


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


cc.en.300.vec:   0%|          | 0.00/4.51G [00:00<?, ?B/s]

In [ ]:
#DO NOT RUN IF USING COLAB


#import sys
#from pathlib import Path
#sys.path.insert(0, str(Path.cwd()))

#import pandas as pd
#import numpy as np
#import torch                       # CNN only
#import cnn_baseline as cnn         # CNN only

# Running this will import FastText vector file, stored on HuggingFace and is >4gb.
#from config import DATA_DIR, FASTTEXT_PATH, TARGETS

#from preprocess import preprocess  # CNN only
#from metrics import compute_metrics, print_confusion_matrix, print_sklearn_report, error_analysis, print_report

#DEVICE = torch.device(             # CNN only
 #   "mps"  if torch.backends.mps.is_available()  else
  #  "cuda" if torch.cuda.is_available()           else
   # "cpu"
#)
#print(f"Device: {DEVICE}")         # CNN only
#print(f"Targets: {TARGETS}")

In [ ]:

# Load data (shared across all models)
train_rows = cnn.load_csv(DATA_DIR / "mis_df_train.csv")
dev_rows   = cnn.load_csv(DATA_DIR / "mis_df_dev.csv")
print(f"Train: {len(train_rows)} | Dev: {len(dev_rows)}")

Train: 600 | Dev: 200


In [ ]:
import pathlib
import sys

# Manually add Path to the builtins or the sys module
# so the script can see it when it loads
import builtins
builtins.Path = pathlib.Path

import cnn_mtl_no_ling as cnn_mtl_pos

## Run models


In [ ]:
# Each model should produce (preds, labels, probs) and store them in `results`.
# Add new models below following the same pattern.
results = {}

#added metrics_cache for later use in ordering all models by Macro F1 and making easy displays
metrics_cache = {}


In [ ]:
import joblib, pickle, torch.nn as nn
from pathlib import Path

SAVE_DIR = SPRINT_DIR / "saved_models"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

def save_checkpoint(key, model=None, result=None):
    """Save model weights and/or (preds, labels, probs) result tuple."""
    safe_key = key.replace(" ", "_").replace("/", "-").replace("(", "").replace(")", "").replace("+", "plus")
    saved = []

    if model is not None:
        if isinstance(model, nn.Module):
            torch.save(model.state_dict(), SAVE_DIR / f"{safe_key}.pt")
            saved.append("state_dict")
        else:
            joblib.dump(model, SAVE_DIR / f"{safe_key}.joblib")
            saved.append("joblib")

    if result is not None:
        with open(SAVE_DIR / f"{safe_key}_result.pkl", "wb") as f:
            pickle.dump(result, f)
        saved.append("result")

    if saved:
        print(f"  [saved] {safe_key} ({', '.join(saved)})")

In [ ]:
# Model name constants — define once, use everywhere
M_TEXT_CNN                 = "TextCNN"
M_TEXT_CNN_TRANSFER        = "TextCNN Transfer"
M_TEXT_CNN_TRANSFER_SHARED = "TextCNN_Transfer_shared"
M_CNN_MTL_LING             = "TextCNN MTL (POS + linguistic)"
M_CNN_MTL_POS              = "TextCNN MTL (POS)"
M_LOGREG                   = "LogReg"
M_LOGREG_EMBED             = "LogReg (+ embeddings)"
M_LOGREG_MTL_POS           = "LogReg MTL (cascaded POS)"

M_ENSEMBLE_SOFT_VOTE       = "Soft Vote Ensemble"
M_ENSEMBLE_MOTIVATED       = "Motivated Ensemble"
M_ENSEMBLE_LEAN_SOFT_VOTE  = "Lean Soft Vote"
M_ENSEMBLE_LEAN_MOTIVATED  = "Lean Motivated Ensemble"

CNN_MODELS      = [M_TEXT_CNN, M_TEXT_CNN_TRANSFER, M_CNN_MTL_LING, M_CNN_MTL_POS]
LR_MODELS       = [M_LOGREG, M_LOGREG_EMBED, M_LOGREG_MTL_POS]
BASE_MODELS     = LR_MODELS + CNN_MODELS
ENSEMBLE_MODELS = [M_ENSEMBLE_SOFT_VOTE, M_ENSEMBLE_MOTIVATED,
                   M_ENSEMBLE_LEAN_SOFT_VOTE, M_ENSEMBLE_LEAN_MOTIVATED]


def rank_key(metrics_key):
    """Sort key for model selection: primary = Macro F1, tiebreaker = AUC-ROC."""
    m = metrics_cache[metrics_key]
    return (m["macro_f1"], m.get("auc_roc", 0.0))


def display_metrics(key_prefix):
    """Display a metrics table for one ensemble/model prefix across all TARGETS."""
    pd.set_option("display.float_format", "{:.4f}".format)
    rows = []
    for t in TARGETS:
        key = f"{key_prefix} — {t}"
        m = metrics_cache[key]
        rows.append({
            "Model": key,
            "Macro F1": m["macro_f1"],
            "F1 (not-op)": m["f1_class0"],
            "F1 (opinion)": m["f1_class1"],
            "F1.5 (recall-weighted)": m["fbeta_class1"],
            "AUC-ROC": m.get("auc_roc", float("nan")),
        })
    return pd.DataFrame(rows).set_index("Model")

#### CNN Baseline

In [ ]:
#CNN Baseline
vocab        = cnn.build_vocab(train_rows, preprocess)
embed_matrix = cnn.load_fasttext_vectors(FASTTEXT_PATH, vocab)
train_loader = cnn.make_loader(train_rows, vocab, shuffle=True,  tokenize_fn=preprocess)
dev_loader   = cnn.make_loader(dev_rows,   vocab, shuffle=False, tokenize_fn=preprocess)

for target in TARGETS:
    model = cnn.TextCNN(len(vocab), embed_matrix).to(DEVICE)
    model = cnn.train_model(model, train_loader, dev_loader, train_rows, DEVICE,
                            train_targets=[target])
    key = f"{M_TEXT_CNN} — {target}"
    results[key] = cnn.predict(model, dev_loader, DEVICE, target=target)
    metrics_cache[key] = compute_metrics(*results[key])
    save_checkpoint(key, model=model, result=results[key])

#### CNN Transfer Learning

In [ ]:
# CNN Transfer Learning
cnn_model = cnn.TextCNN(len(vocab), embed_matrix).to(DEVICE)
cnn_model = cnn.train_model(cnn_model, train_loader, dev_loader, train_rows, DEVICE)
save_checkpoint(M_TEXT_CNN_TRANSFER_SHARED, model=cnn_model)

for target in TARGETS:
    key = f"{M_TEXT_CNN_TRANSFER} — {target}"
    results[key] = cnn.predict(cnn_model, dev_loader, DEVICE, target=target)
    metrics_cache[key] = compute_metrics(*results[key])
    save_checkpoint(key, result=results[key])

#### CNN MTL — POS & Linguistic Features


In [ ]:
import cnn_mtl_ling as cnn_mtl_ling

train_loader_ling = cnn_mtl_ling.make_loader(train_rows, vocab, shuffle=True,  tokenize_fn=preprocess)
dev_loader_ling   = cnn_mtl_ling.make_loader(dev_rows,   vocab, shuffle=False, tokenize_fn=preprocess)

for target in TARGETS:
    model = cnn_mtl_ling.TextCNN(len(vocab), embed_matrix).to(DEVICE)
    model = cnn_mtl_ling.train_model(model, train_loader_ling, dev_loader_ling, train_rows, DEVICE,
                                     train_targets=[target])
    key = f"{M_CNN_MTL_LING} — {target}"
    results[key] = cnn_mtl_ling.predict(model, dev_loader_ling, DEVICE, target=target)
    metrics_cache[key] = compute_metrics(*results[key])
    save_checkpoint(key, model=model, result=results[key])

#### CNN MTL - POS Features only


In [ ]:
import cnn_mtl_no_ling as cnn_mtl_pos

train_loader_pos = cnn_mtl_pos.make_loader(train_rows, vocab, shuffle=True,  tokenize_fn=preprocess)
dev_loader_pos   = cnn_mtl_pos.make_loader(dev_rows,   vocab, shuffle=False, tokenize_fn=preprocess)

for target in TARGETS:
    model = cnn_mtl_pos.TextCNN(len(vocab), embed_matrix).to(DEVICE)
    model = cnn_mtl_pos.train_model(model, train_loader_pos, dev_loader_pos, train_rows, DEVICE,
                                    train_targets=[target])
    key = f"{M_CNN_MTL_POS} — {target}"
    results[key] = cnn_mtl_pos.predict(model, dev_loader_pos, DEVICE, target=target)
    metrics_cache[key] = compute_metrics(*results[key])
    save_checkpoint(key, model=model, result=results[key])

#### Logistic Regression Baseline

In [ ]:
import logreg_baseline as lr

for target in TARGETS:
    key = f"{M_LOGREG} — {target}"
    results[key] = lr.run(train_rows, dev_rows, task=target)
    metrics_cache[key] = compute_metrics(*results[key])
    save_checkpoint(key, result=results[key])

#### Logistic Regression Transfer Learning

In [ ]:
import logreg_transfer as lr_transfer

for target in TARGETS:
    key = f"{M_LOGREG_EMBED} — {target}"
    results[key] = lr_transfer.run(train_rows, dev_rows, task=target)
    metrics_cache[key] = compute_metrics(*results[key])
    save_checkpoint(key, result=results[key])

#### LogReg MTL — Cascaded POS Prediction (Sprint 3)


In [ ]:
import logreg_mtl as lr_mtl

for target in TARGETS:
    key = f"{M_LOGREG_MTL_POS} — {target}"
    results[key] = lr_mtl.run(train_rows, dev_rows, task=target)
    metrics_cache[key] = compute_metrics(*results[key])
    save_checkpoint(key, result=results[key])

### Simple Ensembling

In [ ]:
# Soft vote ensemble
import importlib
import simple_ensemble
importlib.reload(simple_ensemble)
from simple_ensemble import soft_vote

for task in TARGETS:
    preds, labels, probs = soft_vote([results[f"{m} — {task}"] for m in BASE_MODELS])
    key = f"{M_ENSEMBLE_SOFT_VOTE} — {task}"
    results[key] = (preds, labels, probs)
    metrics_cache[key] = compute_metrics(preds, labels, probs)
    save_checkpoint(key, result=results[key])

display_metrics(M_ENSEMBLE_SOFT_VOTE)

### Motivated Ensembling

In [ ]:
# Motivated (F1-weighted) ensemble
import importlib
import motivated_ensemble
importlib.reload(motivated_ensemble)
from motivated_ensemble import motivated_soft_vote

for task in TARGETS:
    f1_weights = [metrics_cache[f"{m} — {task}"]["macro_f1"] for m in BASE_MODELS]
    preds, labels, probs = motivated_soft_vote(
        model_outputs=[results[f"{m} — {task}"] for m in BASE_MODELS],
        f1_weights=f1_weights,
    )
    key = f"{M_ENSEMBLE_MOTIVATED} — {task}"
    results[key] = (preds, labels, probs)
    metrics_cache[key] = compute_metrics(preds, labels, probs)
    save_checkpoint(key, result=results[key])

display_metrics(M_ENSEMBLE_MOTIVATED)

#### Lean Ensemble Soft Vote: uses best logreg and best CNN model only (Sprint 3)

In [ ]:
# Lean soft vote: best LogReg + best CNN per task (selected dynamically from metrics_cache)
best_cnn = {
    task: max(CNN_MODELS, key=lambda m: rank_key(f"{m} — {task}"))
    for task in TARGETS
}
best_logreg = {
    task: max(LR_MODELS, key=lambda m: rank_key(f"{m} — {task}"))
    for task in TARGETS
}

for task in TARGETS:
    print(f"{task}: best LogReg = {best_logreg[task]} | best CNN = {best_cnn[task]}")

for task in TARGETS:
    preds, labels, probs = soft_vote([
        results[f"{best_logreg[task]} — {task}"],
        results[f"{best_cnn[task]} — {task}"],
    ])
    key = f"{M_ENSEMBLE_LEAN_SOFT_VOTE} — {task}"
    results[key] = (preds, labels, probs)
    metrics_cache[key] = compute_metrics(preds, labels, probs)
    save_checkpoint(key, result=results[key])

display_metrics(M_ENSEMBLE_LEAN_SOFT_VOTE)

#### Lean Motivated Ensemble: uses best CNN and best logreg models only (Sprint 3)

In [ ]:
# Lean motivated ensemble: same 2 models, F1-weighted with threshold sweep
for task in TARGETS:
    lean_models = [best_logreg[task], best_cnn[task]]
    f1_weights = [metrics_cache[f"{m} — {task}"]["macro_f1"] for m in lean_models]
    preds, labels, probs = motivated_soft_vote(
        model_outputs=[results[f"{m} — {task}"] for m in lean_models],
        f1_weights=f1_weights,
    )
    key = f"{M_ENSEMBLE_LEAN_MOTIVATED} — {task}"
    results[key] = (preds, labels, probs)
    metrics_cache[key] = compute_metrics(preds, labels, probs)
    save_checkpoint(key, result=results[key])

display_metrics(M_ENSEMBLE_LEAN_MOTIVATED)

### All Results by Target

In [ ]:
# All results by target — everything is now in metrics_cache
all_rows = [{"Model": name, "Macro F1": m["macro_f1"],
             "F1 (not-op)": m["f1_class0"], "F1 (opinion)": m["f1_class1"],
             "F1.5 (recall-weighted)": m["fbeta_class1"],
             "AUC-ROC": m.get("auc_roc", float("nan"))}
            for name, m in metrics_cache.items()]

all_df = pd.DataFrame(all_rows)

for task, label in [("Opinion", "opinion_label"), ("Misinformation", "misinformation_label")]:
    mask = all_df["Model"].str.endswith(f"— {label}")
    df = (all_df[mask]
          .copy()
          .assign(Model=lambda d: d["Model"].str.replace(f" — {label}", "", regex=False))
          .set_index("Model")
          .sort_values("Macro F1", ascending=False))
    print(task, "Ordered by Macro F1 (Desc)")
    print("─" * 60)
    display(df)
    print()

Opinion Ordered by Macro F1 (Desc)
────────────────────────────────────────────────────────────


,Macro F1,F1 (not-op),F1 (opinion),F1.5 (recall-weighted),AUC-ROC
Model,,,,,
Lean Soft Vote,0.7613,0.7911,0.7314,0.7462,0.8027
Lean Motivated Ensemble,0.7613,0.7911,0.7314,0.7462,0.8031
Soft Vote Ensemble,0.7457,0.7788,0.7126,0.7255,0.7943
Motivated Ensemble,0.7457,0.7788,0.7126,0.7255,0.7946
TextCNN,0.7410,0.7733,0.7086,0.7229,0.7992
TextCNN MTL (POS),0.7383,0.7593,0.7174,0.7454,0.7852
TextCNN MTL (POS + linguistic),0.7374,0.7636,0.7111,0.7330,0.7788
TextCNN Transfer,0.7314,0.7623,0.7006,0.7177,0.7827
LogReg (+ embeddings),0.7023,0.7306,0.6740,0.6962,0.7698



Misinformation Ordered by Macro F1 (Desc)
────────────────────────────────────────────────────────────


,Macro F1,F1 (not-op),F1 (opinion),F1.5 (recall-weighted),AUC-ROC
Model,,,,,
Lean Motivated Ensemble,0.9180,0.9553,0.8807,0.8966,0.9644
Lean Soft Vote,0.9090,0.9527,0.8654,0.8654,0.9644
Soft Vote Ensemble,0.9090,0.9527,0.8654,0.8654,0.9669
Motivated Ensemble,0.9090,0.9527,0.8654,0.8654,0.9670
LogReg (+ embeddings),0.8889,0.9428,0.8350,0.8318,0.9501
TextCNN MTL (POS + linguistic),0.8845,0.9388,0.8302,0.8363,0.9553
LogReg MTL (cascaded POS),0.8831,0.9392,0.8269,0.8269,0.9157
TextCNN MTL (POS),0.8816,0.9396,0.8235,0.8174,0.9587
TextCNN,0.8816,0.9396,0.8235,0.8174,0.9591


### Best Model Analysis: Confusion Matrix & Examples

In [ ]:
def print_quadrant_examples(dev_rows, preds, labels, n=3):
    """Print up to n examples from each confusion matrix quadrant."""
    quadrants = {
        "True Positives  (predicted=1, actual=1)": [],
        "True Negatives  (predicted=0, actual=0)": [],
        "False Positives (predicted=1, actual=0)": [],
        "False Negatives (predicted=0, actual=1)": [],
    }
    for row, p, l in zip(dev_rows, preds, labels):
        if   p == 1 and l == 1: quadrants["True Positives  (predicted=1, actual=1)"].append(row)
        elif p == 0 and l == 0: quadrants["True Negatives  (predicted=0, actual=0)"].append(row)
        elif p == 1 and l == 0: quadrants["False Positives (predicted=1, actual=0)"].append(row)
        elif p == 0 and l == 1: quadrants["False Negatives (predicted=0, actual=1)"].append(row)

    for label, rows in quadrants.items():
        print(f"\n── {label} ({len(rows)} total, showing {min(n, len(rows))}) ──")
        for r in rows[:n]:
            print(f"  [{r['id']}] {r['text'][:140]!r}")


for task, label in [("Opinion", "opinion_label"), ("Misinformation", "misinformation_label")]:
    task_keys = [k for k in metrics_cache if k.endswith(f"— {label}")]
    best_key  = max(task_keys, key=rank_key)
    preds, labels_list, probs = results[best_key]
    m = metrics_cache[best_key]

    print(f"\n{'='*60}")
    print(f"{task} — best model: {best_key.replace(f' — {label}', '')}")
    print(f"Macro F1: {m['macro_f1']:.4f}  |  AUC-ROC: {m.get('auc_roc', float('nan')):.4f}")
    print(f"{'='*60}")
    print_confusion_matrix(preds, labels_list)
    print_quadrant_examples(dev_rows, preds, labels_list, n=8)